# Label YouTube Text as either Prediction or Non-Prediction

- **Tasks:**
    1. Load YouTube transcripts extracted from `1-extract_transcripts.ipynb`.
    4. Clean the data.
    5. Create prompt to identify predictions.
    6. Pass each row of text + prompt into the [UF NaviGator Toolkit](https://it.ufl.edu/ai/navigator-toolkit/) or [Groq](https://console.groq.com/home) LLMs.
    7. Perform majority vote (MV) across all LLMs.
    8. Filter by MV label = 1, hence prediction.
    9. Verify that the text is/is not a prediction and update accordingly.

In [1]:
import os
import sys

import re
from typing import List

import pandas as pd

from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))
# from metrics import Metrics
# from feature_extraction import SpacyFeatureExtraction
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory, llm_classify_text

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Load Data

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
yt_data_path = os.path.join(base_data_path, 'yt', 'whisper_transcripts', 'sports')
transcripts = os.listdir(yt_data_path)

In [4]:
dfs = []

for transcript in tqdm(transcripts):
    transcript_path = os.path.join(yt_data_path, transcript)
    df = DataProcessing.load_from_file(path=transcript_path, file_type='csv')
    dfs.append(df)

raw_transcripts_df = DataProcessing.concat_dfs(dfs)
raw_transcripts_df

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00, 50.44it/s]


,Text,Start Time,Duration,Video ID
0,"We all know March Madness is all about filling out and following your bracket,",0.00,3.22,-rjnvL9LL3U
1,"so we offer some first-round advice. Since UMBC beat Virginia in 2018,",3.36,4.30,-rjnvL9LL3U
2,"the 16th seed has won two of 24 meetings versus the one. Occasional upset aside,",7.86,5.06,-rjnvL9LL3U
3,"though, the analytics say advance all the one, two, and three seeds in your bracket.",12.98,3.84,-rjnvL9LL3U
4,"Looking for a long shot? The 11th seed has won half its meetings with the sixth seed of late,",17.18,4.12,-rjnvL9LL3U
...,...,...,...,...
1440,We've got all kinds of great coverage still coming up.,19.72,0.32,ZZN7BAYeOtc
1441,Thanks to everybody for being here.,21.44,0.16,ZZN7BAYeOtc
1442,"Thanks to our incredible staff back in Bristol, Connecticut.",24.26,0.24,ZZN7BAYeOtc
1443,We love you guys.,25.32,0.32,ZZN7BAYeOtc


In [5]:
def clean_data(df) -> list[str]:
    """
    Split running text into sentence-like segments.
    Rules:
    - Split only where a period (.) or question mark (?) is immediately followed by
    whitespace. This avoids splitting decimals like ``4.5`` (digit after the dot,
    not a space).
    - Leading/trailing whitespace is stripped from each segment.
    - Empty segments are dropped.
    Caveat:
    - Abbreviations such as ``Mr. Smith`` still match ". " and may split incorrectly;
    handle those with a tokenizer or an allowlist if needed.
    Parameters
    ----------
    text : dataframe
        Full text to segment (e.g. one block of joined transcript lines).
    Returns
    -------
    list of str
        Non-empty strings, each intended as one sentence or clause.
    """

    text_joined = ' '.join(df.Text.to_list())
    raw_parts = re.split(r'(?<=[.?])\s+', text_joined)
    text_split = [p.strip() for p in raw_parts if p.strip()]

    return text_split

In [6]:
cleaned_transcripts = clean_data(raw_transcripts_df)
cleaned_transcripts

['We all know March Madness is all about filling out and following your bracket, so we offer some first-round advice.',
 'Since UMBC beat Virginia in 2018, the 16th seed has won two of 24 meetings versus the one.',
 'Occasional upset aside, though, the analytics say advance all the one, two, and three seeds in your bracket.',
 'Looking for a long shot?',
 'The 11th seed has won half its meetings with the sixth seed of late, and the ninth seed has won two-thirds of the games against the eighth seed.',
 'Now, you all had UConn winning it last year.',
 'Clark, though, had both finalists.',
 'So, who makes it to the Final Four this time around?',
 'Who cuts down the nets?',
 "You're our clubhouse leader.",
 'Oh, so I get to choose first?',
 'You get to go first.',
 "Well, I tell you, I'm going to take Michigan State, the regular season champs in the Big Ten.",
 'I take them out of the South.',
 "Florida's the most complete team in the field, in my opinion.",
 'I like them coming out of the

In [7]:
cleaned_transcripts[780:810]

["So I'm going the under, and I'm going to go 23-20 Seattle.",
 'Oh, cover by the Patriots.',
 'Good cover.',
 'Patriots plus four and a half.',
 "That's the Madden prediction, by the way.",
 'I did not know that.',
 "Yeah, I would not have known that either, but my table's right next to them, and they had it up on the screen.",
 "We're doing a lot of talking about this.",
 "So here's favored by three and a half to six and a half point in the Super Bowl.",
 'Last five instances, says Hembo.',
 'Obviously, Rams, Patriots, Panthers, and Niners.',
 'In those five games, favorites went 1-4 outright and 0-5 against the spread.',
 'Seahawks sit there at favored by 4.5.',
 'Feels like we have three Patriot covers here by the toxic table into tone.',
 'Dogs have covered five straight and 14 of the last 18, I believe.',
 'Two weeks to prepare for a team.',
 'D-Butt, you were drafted to the New England Patriots.',
 'You thought about putting your bandwagon behind the New England Patriots.',
 "Ha

## Prompt LLM

### Create Prompt

In [8]:
prompt = """
Role:
    You are a text analyst.

Task:
    Determine whether the input text contains a prediction, projection, or forecast.

Definitions:
    - A prediction is a statement that asserts or implies a future outcome, result, or event.
    - It may include confidence, uncertainty, or probability.
    - It must go beyond describing current or past facts.
    - If the sentence only asks who/what/whether and does not state an outcome, it is not a prediction.

Instructions:
    1. Read the text carefully.
    2. Decide if a prediction/projection/forecast is present.
    3. Assign:
        - 1 if a prediction is present
        - 0 if no prediction is present

Output format (strict JSON):
    {
        "label": 0 or 1
    }
"""

### Initialize LLMs

In [9]:
tgmf = TextGenerationModelFactory()

# Option 3: All NaviGator models
# models = tgmf.create_instances(tgmf.get_navigator_model_names())
models = tgmf.create_instances(['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'granite-3.3-8b-instruct'])
models

{'llama-3.1-8b-instant': <text_generation_models.LlamaInstantTextGenerationModel at 0x2244ad10cd0>,
 'llama-3.3-70b-versatile': <text_generation_models.LlamaVersatileTextGenerationModel at 0x2244adc3e90>}

### Pass data + prompt into each LLM

In [12]:
results = []
PROMPT_TYPE = "zero-shot"   # change to "chain-of-thought" when needed

for cleaned_transcript_idx in range(len(cleaned_transcripts)):
    text = cleaned_transcripts[cleaned_transcript_idx]
    print(f"{cleaned_transcript_idx} --- Sentence: {text}")

    for model_name, model in models.items():
        raw_response, llm_label, llm_reasoning = llm_classify_text(
            data=text,
            base_prompt=prompt,
            prompt_type=PROMPT_TYPE,
            model=model
        )

        result = (
            text,
            raw_response,
            llm_label,
            llm_reasoning,
            model_name
        )
        results.append(result)

        if cleaned_transcript_idx < 3:
            if PROMPT_TYPE == "chain-of-thought":
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label} "
                    f"| Reasoning: {llm_reasoning}"
                )
            else:
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label}"
                )

0 --- Sentence: That's the reason you don't have to get frustrated.


 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
1 --- Sentence: It's not even that we have to have you number one in the progression or we have to make sure we call this play for you.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
2 --- Sentence: It's eventually through the flow of the offense, you'll have your chance.
 Model: llama-3.1-8b-instant | Label: 1
 Model: llama-3.3-70b-versatile | Label: 1
3 --- Sentence: And when he does, he seems to always capitalize.
4 --- Sentence: And like Dan says, every route, the stem, the top of the route, They look the same.
5 --- Sentence: And when the movement happens, you separate so much.
6 --- Sentence: He's so talented.
7 --- Sentence: Darnold's 18 completions or 20 or more air yards to Smith and Jigba are the most this season by any QB wide receiver combo.
8 --- Sentence: And we're going to talk about the Patriots' answer to this.
9 --- Sentence: We continue to break down

> - DataFrame below will have the same sentence (in text column) for all three llms (in the llm_name), then next set of rows will be another sentences with same llms.
> - llm_label column: {0: non-prediction, 1: prediction}

In [13]:
results_with_llm_label_df = pd.DataFrame(
    results,
    columns=['text', 'raw_response', 'llm_label', 'llm_reasoning', 'llm_name']
)

# Ensure labels are valid integers (0/1), fallback to -1 if malformed
results_with_llm_label_df['llm_label'] = (
    pd.to_numeric(results_with_llm_label_df['llm_label'], errors='coerce')
      .fillna(-1)
      .astype(int)
)

# Normalize reasoning: keep only when present (chain-of-thought), else None
results_with_llm_label_df['llm_reasoning'] = (
    results_with_llm_label_df['llm_reasoning'].where(
        results_with_llm_label_df['llm_reasoning'].notna(),
        None
    )
)

results_with_llm_label_df.tail(6)

,text,raw_response,llm_label,llm_reasoning,llm_name
450,We love you guys.,"{""label"": 0}",0,None,llama-3.1-8b-instant
451,We love you guys.,"{""label"": 0}",0,None,llama-3.3-70b-versatile
452,And we appreciate everybody for tuning in.,"{""label"": 0}",0,None,llama-3.1-8b-instant
453,And we appreciate everybody for tuning in.,"{""label"": 0}",0,None,llama-3.3-70b-versatile
454,Enjoy Super Bowl 60.,"{""label"": 0}",0,None,llama-3.1-8b-instant
455,Enjoy Super Bowl 60.,"{""label"": 0}",0,None,llama-3.3-70b-versatile


In [14]:
# Reshape to have llm names as column names
results_df = (
    results_with_llm_label_df
    .pivot_table(
        index='text',
        columns='llm_name',
        values='llm_label',
        aggfunc='first'   # one label per model per sentence
    )
    .reset_index()
    .rename(columns={'text': 'Base Sentence'})
)
results_df

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile
0,"A young man by the name of Dwayne Carter, a.k.a.",0,0
1,"All right, are you guys ready for us to make some Super Bowl picks?",0,0
2,"All right, but here's why.",0,0
3,"All right, turnovers.",0,0
4,All right.,0,0
...,...,...,...
220,You talked about them cutting the crossers.,0,0
221,but he might not get the opportunity to affect the game if Seattle is efficient and explosive on first down when they can create that run-pass conflict.,0,1
222,"in the first quarter outscoring their opponents in New England, plus 45.",1,0
223,just the understanding of space and where to go.,0,0


### Majority Vote (across LLMs)

> - llm_label column: {0: non-prediction, 1: prediction}

In [15]:
results_df['Majority Vote Label'] = (
    results_df
    .iloc[:, 1:]               # exclude sentence column
    .mode(axis=1)[0]
    .astype(int)
)

results_df

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,Majority Vote Label
0,"A young man by the name of Dwayne Carter, a.k.a.",0,0,0
1,"All right, are you guys ready for us to make some Super Bowl picks?",0,0,0
2,"All right, but here's why.",0,0,0
3,"All right, turnovers.",0,0,0
4,All right.,0,0,0
...,...,...,...,...
220,You talked about them cutting the crossers.,0,0,0
221,but he might not get the opportunity to affect the game if Seattle is efficient and explosive on first down when they can create that run-pass conflict.,0,1,0
222,"in the first quarter outscoring their opponents in New England, plus 45.",1,0,0
223,just the understanding of space and where to go.,0,0,0


In [18]:
results_df[results_df['Majority Vote Label']==1]

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,Majority Vote Label
4,A Super Bowl champion will be crowned.,1.0,1.0,1
16,"And I just don't see how McDaniels coaching in his ninth, appearing in his tenth Super Bowl with the Drake Mays, he's going to know what to do.",1.0,1.0,1
19,"And I think Hunter Henry is going to catch six, seven passes for maybe 65 yards.",1.0,1.0,1
22,"And I think what you'll see in this game is Clint Kubiak doing that early on, partly to have success, partly to get information.",1.0,1.0,1
30,"And if Iman Worry isn't in the game, then it becomes that much more of a factor.",1.0,1.0,1
54,Andre would be in the slot.,1.0,1.0,1
67,But I think there's even more pressure on Josh McDaniels in scheming up ways to provide the protection for his quarterback because there's a significant drop-off in their points that they've averaged.,1.0,1.0,1
68,"But I will say, one guy that has gone under the radar, even through his success in the end of the regular season, end of the postseason, my X-Factor is going to be Ramondre Stevenson.",1.0,1.0,1
69,But everything you said is exactly why the Patriots will win.,1.0,1.0,1
75,"But my initial prediction when the playoffs were set, I had the New England Patriots beating Seattle.",1.0,1.0,1


In [16]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
save_data_path = os.path.join(base_data_path, "yt", "majority_vote", "sports")
DataProcessing.save_to_file(results_df, path=save_data_path, prefix='batch_4', save_file_type='csv', include_version=False)

Saving CSV file to: c:\Users\adria\OneDrive\Área de Trabalho\UF Data Studio\predictions\prediction_acquition-youtube\../data\yt\majority_vote\sports\batch_4.csv
